# Simple Microsoft Agent Framework Ticket Solver

The same support-ticket project as the three adjacent notebooks: **clean → analyse → decide → reply**. Same sample, prompts, five model calls, and result dictionary. We use the current **Microsoft Agent Framework**, not legacy AutoGen.

```text
message → clean_step → analyse_ticket → add_decision → add_reply → result
```

A **workflow** controls the order. Each **executor** is one async Python step. An **Agent** calls the model inside those steps; it does not choose the workflow order. No routing, retries, memory, streaming, or extra business logic is added.

## Google Colab setup

Use a fresh **Python 3.12 or 3.13 CPU runtime** for each notebook. In the Secrets sidebar, add `OPENAI_API_KEY` and enable **Notebook access** for this notebook. Run cells from top to bottom. If installation requests a restart, restart the session and continue with the imports/credentials cell (do not reinstall repeatedly). No Drive mount or local project files are needed.

The default model is `gpt-4o-mini`. API billing/quota and model access are required; a ChatGPT subscription does not supply API quota. Keys are never printed. Package conflicts can occur if multiple framework notebooks share one runtime—start a fresh session when switching frameworks.


## 1. Install dependencies (Google Colab or Jupyter)

A standard CPU Colab runtime is sufficient; no GPU or Drive mount is needed. Use Python 3.11+ (the `typing.Never` annotation used below requires 3.11). Tested in clean Python 3.12 and 3.13 environments. Colab runtime versions change; this cell prints the actual Python version.

Run the installation cell first, then run the remaining cells in order. If Colab asks for a restart, use **Runtime → Restart session**, then run from this cell again. Restart after changing package versions in a session that has already imported them. If changing only the mode/model, rerun from Section 2 through the sample cell; changing the flag alone does not rebuild the agent.

These pinned versions supply workflows, agents and the OpenAI provider without LangChain or LangGraph. Installation may update shared dependencies in Colab; unrelated preinstalled packages are outside this lesson.

In [ ]:
import sys
if sys.version_info < (3, 11):
    raise RuntimeError("Use a Python 3.11+ notebook runtime.")
print("Python:", sys.version.split()[0])
%pip -q install "agent-framework-core==1.21.0" "agent-framework-openai==1.16.0" "python-dotenv==1.2.4"

## 2. Read your OpenAI API key

In Colab, open the key-shaped **Secrets** panel, confirm the secret is named exactly `OPENAI_API_KEY`, and enable **Notebook access** for this notebook. Then run the cells in order. The sample ticket makes five model requests using your API account.

The loader prefers Colab Secrets, then environment / local `.env`. Outside Colab it can ask through a secure prompt. Keys are never printed.

In [ ]:
MODEL_NAME = "gpt-4o-mini"  # Public OpenAI API model.

In [ ]:
import os
import getpass
from dotenv import load_dotenv, find_dotenv


def read_openai_key():
    # In Colab, always use this notebook's enabled secret.
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        try:
            key = userdata.get("OPENAI_API_KEY")
        except Exception:
            raise RuntimeError(
                "In Colab Secrets, add OPENAI_API_KEY and enable Notebook access, "
                "then rerun this cell."
            ) from None
        if not isinstance(key, str) or not key.strip():
            raise RuntimeError("The OPENAI_API_KEY Colab secret is empty.")
        return key.strip()

    # Local Python fallback; never overwrite an existing environment key.
    try:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv(usecwd=True), override=False)
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        from getpass import getpass
        key = getpass("Enter OPENAI_API_KEY (hidden): ").strip()
    if not key:
        raise ValueError("OPENAI_API_KEY cannot be empty.")
    return key

os.environ["OPENAI_API_KEY"] = read_openai_key()
print("OpenAI key loaded securely; authentication has not yet been tested.")

## 3. Create the Agent

The model name is exactly `gpt-4o-mini`, as in the original LangGraph notebook. Your API provider must support this name; access is not verified here. We preserve the Chat Completions API used by `ChatOpenAI`, through `OpenAIChatCompletionClient`. Optional `OPENAI_BASE_URL` supports an existing compatible endpoint.

`await agent.run(prompt)` returns an `AgentResponse`; `.text.strip()` corresponds to `.content.strip()` in the other notebooks. We pass no session, so the five calls do not share conversation history. No function tools are registered: the existing project has no external tools either. Executors are workflow steps, not model-selected tools.

An API key does not guarantee access or available quota. If the sample reports an authentication error, check the secret; for model-not-found, choose an accessible model consistently in both notebooks if your provider does not support the original name; for quota/rate-limit errors, check API project billing and limits. Provider construction only configures the client; the sample cell makes the actual requests.

In [ ]:
import os
from agent_framework import Agent, WorkflowBuilder, WorkflowContext, executor
from agent_framework.openai import OpenAIChatCompletionClient
from typing import Never

SAMPLE_TICKET = "My account is locked and I have an important meeting in 10 minutes."


client = OpenAIChatCompletionClient(
    model=MODEL_NAME,
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.getenv("OPENAI_BASE_URL") or "https://api.openai.com/v1",
)
agent = Agent(client=client, name="TicketSupport")
print(f"Agent configured for {MODEL_NAME}.")

## 4. Use native workflow shared state

Both this notebook and the LangGraph notebook keep ticket data in framework-managed shared state. Microsoft executors read with `ctx.get_state(key)` and write with `ctx.set_state(key, value)`; these methods are synchronous, so do not `await` them. The workflow owns the state, not a global Python dictionary.

The eventual keys are the same: `ticket`, `analysis`, `decision`, `reply`. Initially the generated fields are absent, as in the LangGraph notebook. The initial customer message arrives as executor input; `clean_step` stores only the cleaned `ticket`. Each later step creates its own output field. There is no prefill with None or empty containers. Each step writes only the field it produces. LangGraph nodes return partial updates that its runtime merges; here we explicitly call `set_state`. There is no LangGraph reducer configured or emulated.

An edge still needs a message to activate the next executor. `ctx.send_message("continue")` is only a control trigger; it carries no ticket data. `WorkflowContext[str]` describes that trigger type. The terminal executor uses `WorkflowContext[Never, dict]` and yields a snapshot of the four shared fields as the final result.

In the pinned framework, writes are staged and committed at superstep boundaries; downstream steps can read them. `get_state` returns a copy, so call `set_state` after changing a retrieved value. This sequential workflow has no concurrent writers; do not assume LangGraph reducer semantics when adding parallel branches.

`TicketState` below is the same executable TypedDict declaration as in the original LangGraph notebook, and is used for the final result and runner annotations. `Optional` permits None values in the type annotation; it does not assign initial values. The original invocation supplies only ticket, and generated fields are written later. The native workflow state is still accessed through `ctx.get_state` / `ctx.set_state`. The type declaration is not passed to `WorkflowBuilder` and does not configure automatic merging, reducers, or runtime validation.

In [ ]:
from typing import TypedDict, Optional

class TicketState(TypedDict):
    ticket: str
    analysis: Optional[dict]
    decision: Optional[str]
    reply: Optional[str]

@executor(id="clean_step")
async def clean_step(message: str, ctx: WorkflowContext[str]) -> None:
    ticket = message.strip()
    if not ticket:
        raise ValueError("Ticket cannot be empty.")
    ctx.set_state("ticket", ticket)
    await ctx.send_message("continue")

## 5. Analyse, decide, reply

The prompts below match the existing notebooks. Analysis calls run sequentially, as in their Python and LangGraph implementations. LCEL uses `RunnableParallel` for these three calls. `async` enables awaiting the framework; it does not make these three calls concurrent.

In [ ]:
@executor(id="analyse_ticket")
async def analyse_ticket(trigger: str, ctx: WorkflowContext[str]) -> None:
    ticket = ctx.get_state("ticket")

    summary = (await agent.run(
        f"Summarize this support ticket in one short sentence:\n{ticket}"
    )).text.strip()

    category = (await agent.run(
        f"Return only one category: billing, technical, or account.\nTicket: {ticket}"
    )).text.strip()

    urgency = (await agent.run(
        f"Return only one urgency: low, medium, or high.\nTicket: {ticket}"
    )).text.strip()

    analysis: dict = {
        "summary": summary,
        "category": category,
        "urgency": urgency,
    }
    ctx.set_state("analysis", analysis)
    await ctx.send_message("continue")

In [ ]:
@executor(id="add_decision")
async def add_decision(trigger: str, ctx: WorkflowContext[str]) -> None:
    ticket = ctx.get_state("ticket")
    analysis = ctx.get_state("analysis")
    prompt = f"""
Review this ticket and the initial analysis:

Ticket:
{ticket}

Analysis:
{analysis}

Give the final decision:
1. Should this ticket be escalated to a human? (Yes/No)
2. Brief reason why.
"""

    decision = (await agent.run(prompt)).text.strip()

    ctx.set_state("decision", decision)
    await ctx.send_message("continue")

In [ ]:
@executor(id="add_reply")
async def add_reply(trigger: str, ctx: WorkflowContext[Never, dict]) -> None:
    ticket = ctx.get_state("ticket")
    decision = ctx.get_state("decision")
    prompt = f"""
You are a customer-support assistant.

Customer message:
{ticket}

Decision:
{decision}

Write a short and helpful reply to the customer.
Do not mention prompts, AI, or internal system details.
"""

    reply = (await agent.run(prompt)).text.strip()

    ctx.set_state("reply", reply)
    result: TicketState = {
        "ticket": ctx.get_state("ticket"),
        "analysis": ctx.get_state("analysis"),
        "decision": ctx.get_state("decision"),
        "reply": ctx.get_state("reply"),
    }
    await ctx.yield_output(result)

## 6. Connect the workflow

Each step uses `@executor(id=...)` above its function. `create_workflow()` connects those decorated executors directly. Executor instances are reused, while each call builds a fresh workflow with its own state store.

Edges control the sequence; they carry only the `"continue"` trigger after cleaning. `set_state` does not schedule the next step by itself. The initial input still contains the customer's message, and only the final output contains the complete result dictionary.

In [ ]:
def create_workflow():
    return (
        WorkflowBuilder(start_executor=clean_step, output_from=[add_reply])
        .add_edge(clean_step, analyse_ticket)
        .add_edge(analyse_ticket, add_decision)
        .add_edge(add_decision, add_reply)
        .build()
    )

## 7. Run ONE function

Jupyter supports top-level `await`. The corresponding call is **`await solve_ticket(message)`**; avoid `asyncio.run()` inside a notebook's already-running event loop. The printed fields and returned dictionary match the other versions. Actual model wording can vary.

In [ ]:
async def solve_ticket(message: str) -> TicketState:
    workflow = create_workflow()
    run_result = await workflow.run(message)
    outputs = run_result.get_outputs()
    if len(outputs) != 1:
        raise RuntimeError(f"Expected one final result; got {len(outputs)}.")
    result = outputs[0]

    print("\n========== TICKET RESULT (Microsoft Agent Framework) ==========")
    print("Message  :", result["ticket"])
    print("\nAnalysis :")
    for key, val in result["analysis"].items():
        print(f"  - {key}: {val}")
    print("\nDecision :\n", result["decision"])
    print("\nReply to customer:\n", result["reply"])
    print("===============================================================")
    return result

In [ ]:
result = await solve_ticket(SAMPLE_TICKET)

## 8. Explain the four implementations

| Concept | Pure Python notebook | LCEL notebook | LangGraph notebook | This Microsoft notebook |
|---|---|---|---|---|
| Order | Direct function calls | `a \| b \| c` | Nodes + edges, `compile()` | Executors + edges, `build()` |
| Model call | `ChatOpenAI.invoke` | Prompt → model → parser | `ChatOpenAI.invoke` inside nodes | `await Agent.run` inside executors |
| State | Variables and final dict | `.assign()` preserves and adds fields | `TypedDict`, returned updates merged | Native workflow state via `get_state` / `set_state` |
| Three analyses | Sequential | Concurrent `RunnableParallel` | Sequential inside one node | Sequential inside one executor |
| Final result | Function returns dict | `.invoke()` returns dict | `.invoke()` returns state | `yield_output` → `get_outputs()` |
| Conversation memory | None used | None used | No checkpointer used | No `AgentSession` used |
| Model-selected tools | None | None | None | None |

**Teaching sequence:** start with ordinary function calls, show LCEL composition, show LangGraph state updates, then show Microsoft shared-state reads/writes and separate control messages. All four are fixed pipelines here, even though the libraries support more advanced agent behavior.

The original LCEL introduction mentions structured output, but its implementation uses text parsers, not schema enforcement. All four preserve plain-text analysis fields and a free-text escalation decision. They recommend escalation; none actually contacts a human or unlocks an account.

**Expected sample shape:** `ticket` is the cleaned original message; `analysis` has `summary`, `category`, `urgency`; `decision` answers Yes/No with a reason; `reply` is customer-facing text. For the sample ticket, account/high/Yes are plausible responses; exact model wording and decisions can vary.

## Official references

API references checked on 10 October 2026, and validated against the installed versions pinned above:

- [Microsoft Agent Framework overview](https://learn.microsoft.com/en-us/agent-framework/overview/agent-framework-overview)
- [OpenAI provider: Agent and Chat Completions client](https://learn.microsoft.com/en-us/agent-framework/integrations/by-component/model-providers/openai)
- [Microsoft's executor and edge example](https://github.com/microsoft/agent-framework/blob/main/python/samples/03-workflows/_start-here/step1_executors_and_edges.py)

Microsoft's current Python naming distinguishes `OpenAIChatClient` (Responses API) from `OpenAIChatCompletionClient` (Chat Completions). This notebook deliberately uses the latter to keep the provider API comparable to the existing project.

### Colab setup references

- [Official Colab Secrets implementation and error types](https://github.com/googlecolab/colabtools/blob/main/google/colab/userdata.py)
- [Colab Python 3.13 runtime announcement](https://github.com/googlecolab/colabtools/issues/6081)
- [OpenAI provider configuration](https://learn.microsoft.com/en-us/agent-framework/integrations/by-component/model-providers/openai)

### Shared-state API reference

Both implementations support shared state. LangGraph applies returned updates; Microsoft uses explicit key writes and separate messages for routing. Each call builds a fresh workflow with its own state store; the decorated executor instances are reused. No persistent checkpoint or conversation session is enabled.

- [Microsoft workflow state documentation](https://learn.microsoft.com/en-us/agent-framework/concepts/workflows/state)
- [Microsoft workflow state source](https://github.com/microsoft/agent-framework/blob/main/python/packages/core/agent_framework/_workflows/_state.py) (also inspected in the installed 1.21.0 package).